In [1]:
import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from tqdm import tqdm

import lightgbm as lgb
import shap

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.metrics import roc_curve

# 윈도우 기본 한글 폰트 설정
mpl.rcParams["font.family"] = "Malgun Gothic"  # 맑은 고딕 사용

# 마이너스 깨짐 방지
mpl.rcParams["axes.unicode_minus"] = False  # 음수 기호 정상 출력

In [8]:
# 데이터 및 분석기 로드
df = pd.read_csv(r'F:\porti\2026\Control_Defines_Difficulty\Control_Defines_Difficulty\3_comment_original\ALL_out\all_merged_comments.csv')
df

,selftext,source,region
0,워프레임 신규 공포 콘텐츠 '그림자 화가' 무료 다운로드,dcinside_aion2,국내
1,그냥 대가리박고 십부장템 너프하죠?,dcinside_aion2,국내
2,마도성 pve좋다 좋다 하는데,dcinside_aion2,국내
3,와 호법은 진짜 초월고단계지옥이네,dcinside_aion2,국내
4,배럭 막았음 좋겠네,dcinside_aion2,국내
...,...,...,...
14928,ㅄ같은 마사지를 보았습니다,youtube,국내
14929,아니 엔진 하나 바꾼다고 이렇게 퀄이 좋냐고!,youtube,국내
14930,lets gooooooo plz don't make same mistakes yo...,youtube,국내
14931,클베 언제인지 아시나요?,youtube,국내


In [ ]:
# 전처리 확인 - 중복, 빈 텍스트, 어느 나라 언어인지 확인 칼럼 추가
# 언어 판별 컬럼 추가
from langdetect import detect

def detect_korean_custom(text):

    if pd.isna(text):
        return "unknown"
    
    text = str(text).strip()

    if text == "":
        return "unknown"

    # 한글 포함은 무조건 한국어
    if re.search(r"[가-힣]", text):
        return "ko"

    # 디시식 표현
    dc_patterns = ["ㅇ", "ㅁㅌㅊ", "ㅅㅂ", "ㄹㅇ", "ㅇㅇ", "ㅇㅅㅇ", "ㄱㄱ", "ㅎ", "ㅎㅎㅎ"]
    for p in dc_patterns:
        if p in text:
            return "ko"

    # 영어만 있는 경우
    if re.fullmatch(r"[a-zA-Z0-9\s\W]+", text):
        return "en"

    # fallback (기타 언어)
    try:
        if text.strip() == " ":
            return "unknown"
        return detect(text)
    except:
        return "error"


df["language"] = df["selftext"].apply(detect_korean_custom)

print("\n언어 분포:")
print(df["language"].value_counts())


언어 분포:
ko         14135
en           712
ru            36
es             8
pt             8
fr             6
tr             4
ar             4
zh-cn          4
unknown        4
fa             3
bg             3
uk             2
ja             1
mk             1
sl             1
hr             1
Name: language, dtype: int64


In [13]:
print(f"전체 데이터 수: {len(df)}")

# 중복 조회
duplicates = df[df.duplicated(subset=["selftext","source","region"], keep=False)]
print(f"\n중복 데이터 수: {len(duplicates)}")
print(duplicates.head())

# 빈 텍스트 조회
df["selftext"] = df["selftext"].fillna("").astype(str)
empty_rows = df[df["selftext"].str.strip() == ""]
print(f"\n빈 텍스트 수: {len(empty_rows)}")
print(empty_rows.head())

# 링크 포함 조회
link_rows = df[df["selftext"].str.contains("http|https", case=False, na=False)]
print(f"\n링크 포함 데이터 수: {len(link_rows)}")
print(link_rows.head())

전체 데이터 수: 14933

중복 데이터 수: 0
Empty DataFrame
Columns: [selftext, source, region, language]
Index: []

빈 텍스트 수: 4
     selftext           source region language
6774               inven_aion2     국내  unknown
6981             reddit_MMORPG     해외  unknown
7048           reddit_Aion2Hub     해외  unknown
7062              reddit_Aion2     해외  unknown

링크 포함 데이터 수: 0
Empty DataFrame
Columns: [selftext, source, region, language]
Index: []


In [14]:
# 빈 텍스트 제거
df["selftext"] = df["selftext"].fillna("").astype(str)
df = df[df["selftext"].str.strip() != ""]

print(f"\n빈 텍스트 제거 후 데이터 수: {len(df)}")


빈 텍스트 제거 후 데이터 수: 14929


In [15]:
# 언어 분포 (카운트 + 비중)
lang_counts = df["language"].value_counts()
lang_ratio = df["language"].value_counts(normalize=True) * 100

lang_summary = pd.DataFrame({
    "count": lang_counts,
    "ratio(%)": lang_ratio.round(2)
})

print("\n언어 분포:")
print(lang_summary)

# 한국어 아닌 데이터 조회
non_korean = df[df["language"] != "ko"]

print(f"\n한국어 아닌 데이터 수: {len(non_korean)} ({len(non_korean)/len(df)*100:.2f}%)")
non_korean


언어 분포:
       count  ratio(%)
ko     14135     94.68
en       712      4.77
ru        36      0.24
pt         8      0.05
es         8      0.05
fr         6      0.04
zh-cn      4      0.03
ar         4      0.03
tr         4      0.03
fa         3      0.02
bg         3      0.02
uk         2      0.01
ja         1      0.01
mk         1      0.01
sl         1      0.01
hr         1      0.01

한국어 아닌 데이터 수: 794 (5.32%)


,selftext,source,region,language
4594,131,dcinside_aion2,국내,en
4602,xx,dcinside_aion2,국내,en
5195,PTSD,dcinside_lostarkm,국내,en
5797,5090,dcinside_lostarkm,국내,en
5927,S25U 16G,dcinside_lostarkm,국내,en
...,...,...,...,...
14808,I will never play this on Mobile.,youtube,국내,en
14810,just another gacha mmo,youtube,국내,en
14820,Hope the mobile version makes u to progress th...,youtube,국내,en
14824,هتكسر الدنيا على الموبايل اللعبه,youtube,국내,ar


In [17]:
# 번역 테스트용 저장
output_path = "merged_comments_global_translate.csv"
non_korean.to_csv(output_path, index=False, encoding="utf-8-sig")

In [19]:
# 한국어 비중이 약 95%로 한국어 댓글만 사용해도 합리하다고 생각
# 한국어만 필터링
df_ko = df[df["language"] == "ko"]
print(f"한국어 데이터 수: {len(df_ko)} ({len(df_ko)/len(df)*100:.2f}%)")

# 저장
OUTPUT_PATH = "all_merged_comments_korean_only.csv"
df_ko.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print(f"\n한국어 데이터 저장 완료: {OUTPUT_PATH}")
df_ko

한국어 데이터 수: 14135 (94.68%)

한국어 데이터 저장 완료: all_merged_comments_korean_only.csv


,selftext,source,region,language
0,워프레임 신규 공포 콘텐츠 '그림자 화가' 무료 다운로드,dcinside_aion2,국내,ko
1,그냥 대가리박고 십부장템 너프하죠?,dcinside_aion2,국내,ko
2,마도성 pve좋다 좋다 하는데,dcinside_aion2,국내,ko
3,와 호법은 진짜 초월고단계지옥이네,dcinside_aion2,국내,ko
4,배럭 막았음 좋겠네,dcinside_aion2,국내,ko
...,...,...,...,...
14927,아크라시아가 멸망하는 운명,youtube,국내,ko
14928,ㅄ같은 마사지를 보았습니다,youtube,국내,ko
14929,아니 엔진 하나 바꾼다고 이렇게 퀄이 좋냐고!,youtube,국내,ko
14931,클베 언제인지 아시나요?,youtube,국내,ko


In [38]:
df = df_ko.copy()

In [235]:
keywords = {
    "난이도_높음": [
        "추락", "어렵", "힘들", "빡세", "헬", "스트레스", "지옥", "고통", "다터졌네", "만점", 
        "쉽지않", "레벨링", "썰자팟", "지옥", "초월", "고단계", "숨이 막히", "피로도",
        "원거리쟁","못잡", "미친놈"
    ],
    "난이도_낮음": [
        "쩔", "공유", "솔플", "무난", "촌섭", "편함", "할만", "쉬움", "널널", "버스", 
        "주작", "빨리올리는", "꿀팁", "개사기네", "무한힐", "효율", "딸깍", "개꿀",
        "금방이네","편하","평타만"
    ],
    "조작_복잡": [
        "시프트", "조작", "컨트롤", "손", "피로", "바쁘", "복잡", "손아픔", "손꼬임", 
        "피지컬", "연타", "멀티태스킹", "정신없", "클릭", "터치", "옵션"
    ],
    "조작_간단": [
        "간단", "편함", "단순", "쉬운조작", "편한조작", "직관적", "적응쉬움", "금방익힘", 
        "편의성", "자동", "원터치"
    ],
    "입력_과다": [
        "버튼 많", "입력 많", "손 많이", "눌러야", "조작 많", "복잡 입력", "버튼빡셈", 
        "손바쁨", "광클", "연사", "키보드박살"
    ],
    "입력_적음": [
        "버튼 적", "간단 입력", "조작 적", "입력 적", "간편", "편한 입력", "조작 최소", 
        "부담없음", "간결"
    ],
    "반응_실패": [
        "못피", "늦", "반응 못", "못피함", "대응 못", "반응 느림", "타이밍 실패", "못막", "놓침"
    ],
    "반응_성공": [
        "피했", "잘 피", "회피 성공", "잘막", "대응 가능", "컨 성공", "피하기 쉬움", "회피 잘됨", "컨 잘됨",
        "잘한거임", "몇트"
    ],
    "전투_실패": [
        "어글", "피격", "죽", "터짐", "녹음", "순삭", "한방", "끔살", "삭제됨", "너프", "거지같", "밸런스붕괴",
        "못깨요", "못잡"
    ],
    "전투_성공": [
        "안맞", "버팀", "살았", "생존", "안죽", "버티기", "유지됨", "클리어", "안맞음", "보상", "성능", "사기급",
        "던전 패싱", "레이드"
    ],
    "자동화_의존": [
        "매크로", "오토", "자동사냥", "자동", "어뷰", "어뷰징", "버그", "자동플레이", "오토플", "딸깍겜",
        "메크로"
    ],
    "수동_플레이": [
        "시프트", "수동", "직접", "손컨", "수동플", "직접조작", "몰이사냥", "직접함", "컨트롤함", "손맛",
        "시공", "봉던", "원정", "채집 실패", "탐험"
    ],
    "부정_경험": [
        "던져버리노", "틀딱들", "손봐야", "개버러지", "씹버러지", "틀딱", "못만들", "실망", "짜증", 
        "불만", "답도 없", "추방", "우습다", "쫓겨났음", "쫒겨났음", "최적화", "렉", "발열", "그래픽왜이럼", "망겜",
        "노안", "다운증후군","재미없", "모자라", "잘못됐음", "안됌", "에바", "또라이", "망함", "소통없",
        "뭐함", "미침", "노잼", "스킵 안되는", "크래시", "구림", "부족하"
    ],
    "긍정_경험": [
        "재밌노", "신기한데", "재미떠", "꿀잼", "존나재밌음", "갓겜", "재밌다", "추천", "좋다", 
        "잘만들긴했다", "할만하", "재밌겠다", "괜찮", "발전했네", "기대됨", "혜자", "귀엽", "좋네",
        "개추", "재밌는데", "기대가 됩니다", "맘에들긴함","편하", "웅장"
    ]
}

In [236]:
# 계산 그룹 정의
pos_cols = ["난이도_낮음", "조작_간단", "입력_적음", "반응_성공", "전투_성공", "수동_플레이", "긍정_경험"]
neg_cols = ["난이도_높음", "조작_복잡", "입력_과다", "반응_실패", "전투_실패", "자동화_의존", "부정_경험"]

#  행별 분석 함수
def analyze_row(text):
    text = str(text).lower()
    counts = {k: sum(1 for w in words if re.search(re.escape(w), text)) for k, words in keywords.items()}
    
    p_sum = sum(counts[col] for col in pos_cols)
    n_sum = sum(counts[col] for col in neg_cols)
    total = p_sum + n_sum
    
    if total == 0:
        return pd.Series(["중립", "기타", "0.00%", "100.00%", "0.00%"], 
                         index=["sentiment", "category", "positive_ratio", "neutral_ratio", "negative_ratio"])
    
    label = "긍정" if p_sum > n_sum else ("부정" if n_sum > p_sum else "중립")
    category = max(counts, key=counts.get)
    
    return pd.Series([label, category, f"{(p_sum/total*100):.2f}%", "0.00%", f"{(n_sum/total*100):.2f}%"], 
                     index=["sentiment", "category", "positive_ratio", "neutral_ratio", "negative_ratio"])

# 실행 및 결과 합치기
# 기존 결과 컬럼 삭제 후 새로 계산
df = df.drop(columns=["sentiment", "category", "positive_ratio", "neutral_ratio", "negative_ratio"], errors='ignore')
analysis_results = df["selftext"].apply(analyze_row)
df = pd.concat([df, analysis_results], axis=1)

# 결과 출력창
print("\n" + "="*95)
print(f"{'댓글 내용 (selftext)':<50} | {'감정':<4} | {'긍정%':<8} | {'부정%':<8}")
print("-" * 95)

# 상위 15개 샘플 출력
for _, r in df.head(15).iterrows():
    txt = str(r['selftext']).replace('\n', ' ')[:48]
    print(f"{txt:<50} | {r['sentiment']:<4} | {r['positive_ratio']:<8} | {r['negative_ratio']:<8}")

print("="*95)

# 전체 통계
sent_val = df["sentiment"].value_counts(normalize=True) * 100
print(f"[전체 요약]  긍정: {sent_val.get('긍정',0):.2f}% | 부정: {sent_val.get('부정',0):.2f}% | 중립: {sent_val.get('중립',0):.2f}%")
print(f"Complaint Score: {sent_val.get('부정',0):.2f}")
print("="*95)


댓글 내용 (selftext)                                   | 감정   | 긍정%      | 부정%     
-----------------------------------------------------------------------------------------------
워프레임 신규 공포 콘텐츠 '그림자 화가' 무료 다운로드                    | 중립   | 0.00%    | 0.00%   
그냥 대가리박고 십부장템 너프하죠?                                | 부정   | 0.00%    | 100.00% 
마도성 pve좋다 좋다 하는데                                   | 긍정   | 100.00%  | 0.00%   
와 호법은 진짜 초월고단계지옥이네                                 | 부정   | 0.00%    | 100.00% 
배럭 막았음 좋겠네                                         | 중립   | 0.00%    | 0.00%   
원정 큐브안까고 뺑빼이 돌아도댐?                                 | 긍정   | 100.00%  | 0.00%   
커마 ㅁㅌㅊ?                                            | 중립   | 0.00%    | 0.00%   
왜 로리커마 90%는 다운증후군같이 만드는거냐?                         | 부정   | 0.00%    | 100.00% 
부활석은 지금이 밸런스가 최고 좋음                                | 중립   | 0.00%    | 0.00%   
모든 살성이 다 pve 1.5인분 하는건 아니다                         | 중립   | 0.00%    | 0.00%   
국민 매크로 게임이 된건 궁성짤 주인공이 

In [237]:
print("\n" + "="*50)
print("[전체 통계 요약]")
print("="*50)
sent_counts = df["sentiment"].value_counts(normalize=True) * 100
for label, ratio in sent_counts.items():
    print(f"{label} 비중: {ratio:.2f}%")

complaint_score = (df["sentiment"] == "부정").mean() * 100
print(f"Final Complaint Score: {complaint_score:.2f}")
print("="*50)


[전체 통계 요약]
중립 비중: 82.00%
부정 비중: 9.83%
긍정 비중: 8.17%
Final Complaint Score: 9.83


In [238]:
df

,selftext,source,region,language,sentiment,category,positive_ratio,neutral_ratio,negative_ratio
0,워프레임 신규 공포 콘텐츠 '그림자 화가' 무료 다운로드,dcinside_aion2,국내,ko,중립,기타,0.00%,100.00%,0.00%
1,그냥 대가리박고 십부장템 너프하죠?,dcinside_aion2,국내,ko,부정,전투_실패,0.00%,0.00%,100.00%
2,마도성 pve좋다 좋다 하는데,dcinside_aion2,국내,ko,긍정,긍정_경험,100.00%,0.00%,0.00%
3,와 호법은 진짜 초월고단계지옥이네,dcinside_aion2,국내,ko,부정,난이도_높음,0.00%,0.00%,100.00%
4,배럭 막았음 좋겠네,dcinside_aion2,국내,ko,중립,기타,0.00%,100.00%,0.00%
...,...,...,...,...,...,...,...,...,...
14927,아크라시아가 멸망하는 운명,youtube,국내,ko,중립,기타,0.00%,100.00%,0.00%
14928,ㅄ같은 마사지를 보았습니다,youtube,국내,ko,중립,기타,0.00%,100.00%,0.00%
14929,아니 엔진 하나 바꾼다고 이렇게 퀄이 좋냐고!,youtube,국내,ko,중립,기타,0.00%,100.00%,0.00%
14931,클베 언제인지 아시나요?,youtube,국내,ko,중립,기타,0.00%,100.00%,0.00%


In [239]:
df_neutral = df[df["category"] == "기타"].copy()

In [240]:
df_neutral

,selftext,source,region,language,sentiment,category,positive_ratio,neutral_ratio,negative_ratio
0,워프레임 신규 공포 콘텐츠 '그림자 화가' 무료 다운로드,dcinside_aion2,국내,ko,중립,기타,0.00%,100.00%,0.00%
4,배럭 막았음 좋겠네,dcinside_aion2,국내,ko,중립,기타,0.00%,100.00%,0.00%
6,커마 ㅁㅌㅊ?,dcinside_aion2,국내,ko,중립,기타,0.00%,100.00%,0.00%
8,부활석은 지금이 밸런스가 최고 좋음,dcinside_aion2,국내,ko,중립,기타,0.00%,100.00%,0.00%
9,모든 살성이 다 pve 1.5인분 하는건 아니다,dcinside_aion2,국내,ko,중립,기타,0.00%,100.00%,0.00%
...,...,...,...,...,...,...,...,...,...
14927,아크라시아가 멸망하는 운명,youtube,국내,ko,중립,기타,0.00%,100.00%,0.00%
14928,ㅄ같은 마사지를 보았습니다,youtube,국내,ko,중립,기타,0.00%,100.00%,0.00%
14929,아니 엔진 하나 바꾼다고 이렇게 퀄이 좋냐고!,youtube,국내,ko,중립,기타,0.00%,100.00%,0.00%
14931,클베 언제인지 아시나요?,youtube,국내,ko,중립,기타,0.00%,100.00%,0.00%


In [190]:
df_etc = df_neutral.copy()

In [191]:
def extract_words(text):
    # 한글, 영문, 숫자만 남기고 특수문자 제거
    text = re.sub(r'[^가-힣a-zA-Z0-9\s]', ' ', str(text))
    # 공백 기준으로 단어 분리 (2글자 이상인 단어만 추출)
    words = [word for word in text.split() if len(word) >= 2]
    return words

# 3. 모든 단어를 하나의 리스트로 통합
all_etc_words = []
for text in df_etc["selftext"]:
    all_etc_words.extend(extract_words(text))

# 4. 단어 빈도수 계산 및 내림차순 정렬
word_counts = Counter(all_etc_words)
# 상위 50개 출력
most_common_words = word_counts.most_common(50)

# 5. 결과 출력
print("\n" + "="*50)
print(f"🔍 [기타 카테고리] 단어 빈도 TOP 50 (총 {len(df_etc)}건 분석)")
print("-" * 50)
print(f"{'단어':<15} | {'빈도수':<10}")
print("-" * 50)

for word, count in most_common_words:
    print(f"{word:<15} | {count:<10}")

print("="*50)


🔍 [기타 카테고리] 단어 빈도 TOP 50 (총 11583건 분석)
--------------------------------------------------
단어              | 빈도수       
--------------------------------------------------
진짜              | 445       
모바일             | 367       
그냥              | 362       
로아              | 312       
게임              | 310       
이거              | 286       
너무              | 254       
근데              | 225       
아이온             | 180       
지금              | 171       
아이온2            | 165       
아니              | 152       
이제              | 149       
제발              | 143       
모바일로            | 127       
게임을             | 122       
있습니다            | 122       
게임이             | 118       
있는              | 115       
정말              | 111       
다른              | 107       
나는              | 105       
이게              | 103       
같은              | 102       
하고              | 102       
많이              | 100       
모든              | 97        
작업장             | 96        
다시              |

In [ ]:
# -----------------------------
# 6️⃣ 저장
# -----------------------------
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print(f"\n최종 저장 완료: {OUTPUT_PATH}")
